In [0]:
SHOW TABLES IN bronze_olist;

In [0]:
USE SCHEMA bronze_olist;

In [0]:
DROP SCHEMA IF EXISTS silver_olist CASCADE; --Remove schema if there is
CREATE SCHEMA IF NOT EXISTS silver_olist;--Create new silver schema

##### 1. Table Orders

In [0]:
--View table Orders
SELECT * FROM orders LIMIT 5;

In [0]:
--Check if there is null value for order_id OR order_purchase_timestamp
SELECT *
FROM orders
WHERE
  order_id IS NULL
  OR
  order_purchase_timestamp IS NULL
--No row returned


In [0]:
-- Check whether each order is unique
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT order_id) AS distinct_rows,
  (COUNT(*) = COUNT(DISTINCT order_id)) AS is_unique
FROM orders;


In [0]:
--Change column types & extract dates, then load to silver layer

CREATE OR REPLACE TABLE silver_olist.orders AS 
  SELECT 
    order_id,
    customer_id,
    order_status,
    CAST(order_purchase_timestamp AS TIMESTAMP) as order_purchase_ts,
    TO_DATE(order_purchase_timestamp) as order_purchase_date,
    CAST(order_approved_at AS TIMESTAMP) as order_approved_ts,
    TO_DATE(order_approved_at) as order_approved_date,
    CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_ts,
    TO_DATE(order_delivered_carrier_date) as order_delivered_carrier_date,
    CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_ts,
    TO_DATE(order_delivered_customer_date) as order_delivered_customer_date,
    TO_DATE(order_estimated_delivery_date) as order_estimated_delivery_date
  FROM orders;

#####2. Table order_items

In [0]:
SELECT * FROM order_items

In [0]:
--Check if each combination of order_item and order_id is unique. Each combination is unique.
SELECT COUNT(*),
      COUNT(DISTINCT order_item_id, order_id),
      (COUNT(*) = COUNT(DISTINCT order_item_id, order_id)) AS is_unique 
      FROM order_items

In [0]:
--Check if there is null value for order_id, order_item_id, shipping_limit_date, price, freight_value. There is no null value.
SELECT * 
FROM order_items
WHERE
  order_id IS NULL
  OR
  order_item_id IS NULL
  OR
  shipping_limit_date IS NULL
  OR
  price IS NULL
  OR
  freight_value IS NULL

In [0]:
CREATE OR REPLACE TABLE silver_olist.order_items AS
  SELECT 
    order_id,
    product_id, 
    order_item_id,
    seller_id,
    CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_ts,
    TO_DATE(shipping_limit_date) AS shipping_limit_date,
    CAST(price AS DOUBLE) AS price,
    CAST(freight_value AS DOUBLE) AS freight_value
  FROM order_items


#####3. Table order_payments

In [0]:
SELECT * FROM order_payments LIMIT 5
-- View table

In [0]:
--Check if each combincation of order_id and payment_sequential is unique. Each combination is unique.
SELECT COUNT(*),
      COUNT(DISTINCT order_id,payment_sequential),
      (COUNT(*) = COUNT(DISTINCT order_id,payment_sequential)) AS is_unique
FROM order_payments

In [0]:
--Check whether null table is in the payment. No null value returned.
SELECT 
  * 
FROM 
  order_payments 
WHERE 
  payment_sequential IS NULL
  OR
  order_id IS NULL
  OR
  payment_type IS NULL
  OR
  payment_installments IS NULL
  OR  
  payment_value IS NULL

In [0]:
--Check payment_type
SELECT DISTINCT payment_type FROM order_payments

In [0]:
--Cast values to right type
--Put to silver layer
CREATE OR REPLACE TABLE silver_olist.order_payments AS
  SELECT 
    order_id,
    CAST(payment_sequential AS INT) AS payment_sequential,
    INITCAP(REPLACE(payment_type,'_',' ')) AS payment_type,
    CAST(payment_installments AS INT) AS payment_installments,
    CAST(payment_value AS DOUBLE) AS payment_value  FROM order_payments
  

#####4. Table order_reviews

In [0]:
--Count rows of order_reviews table, and check whether each review_id is unique
SELECT COUNT(*), count(distinct review_id) FROM order_reviews
--There are 104162 rows, and 102957 distinct review_id. There may be duplicated review_id. 

In [0]:
--View columns of the table.
SELECT * FROM order_reviews limit 5

In [0]:
--Check null values for order_id column
SELECT * 
FROM order_reviews 
WHERE order_id IS NULL
--There are many rows with null order_id, in those rows, review_score is null too

In [0]:
--Check null values for review_id
SELECT * 
FROM order_reviews 
WHERE review_id IS NULL
--There is 1 row that review_id is null, and review score is not integer


In [0]:
--Check null values for review_score
SELECT * 
FROM order_reviews 
WHERE review_score IS NULL
--There are about 2380 rows with null review_score.

In [0]:
-- Check for records where review_score is not an integer
SELECT *
FROM order_reviews
WHERE 
  TRY_CAST(review_score AS INT) IS NULL;
--About 5000 rows where review score is not an integer

In [0]:
--Check if each review_id is unique after removing null records review_id, order_id,review_score and removing non-integer record of review_score
WITH new_reviews AS(
  SELECT 
      review_id,
    order_id,
    CAST(review_score AS INT) AS review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp 
  FROM order_reviews
  WHERE
      order_id IS NOT NULL
    OR review_id IS NOT NULL
    OR review_score IS NOT NULL 
    OR TRY_CAST(review_score AS INT) IS NOT NULL)
    SELECT COUNT(*), count(distinct review_id) FROM new_reviews

In [0]:
--cte not_null_review_score: remove records in which review_score is null from order_reviews table
--cte integer_review_score: leave out records having non-integer review_score
--cte not_null_review_id: remove records in which review_id is null from cte integer_review_score
WITH not_null_review_score AS(
  SELECT 
    *
  FROM
  order_reviews
  WHERE review_score IS NOT NULL
),
integer_review_score AS(
  SELECT 
    * 
  FROM 
  not_null_review_score 
  WHERE 
  TRY_CAST(review_score AS INT) IS NOT NULL),
not_null_review_id AS(
  SELECT * 
  FROM
  integer_review_score 
  WHERE review_id IS NOT NULL
)
--Check whether there is record having null order_id 
SELECT * 
FROM not_null_review_score
WHERE order_id IS NULL
--There is no more record having null order_id



In [0]:
--Check if there is unique record for each review after excluding unexpected values
--There are 99225 rows but 98411 distinct review_id. So, there is duplicated review_id
WITH not_null_review_score AS(
  SELECT 
    *
  FROM
  order_reviews
  WHERE review_score IS NOT NULL
),
integer_review_score AS(
  SELECT 
    * 
  FROM 
  not_null_review_score 
  WHERE 
  TRY_CAST(review_score AS INT) IS NOT NULL),
not_null_review_id AS(
  SELECT * 
  FROM
  integer_review_score
  WHERE review_id IS NOT NULL
)
SELECT COUNT(*), 
        COUNT(distinct review_id)
FROM
not_null_review_id


In [0]:
--Looking at count value of review_id, remove duplicates by row_number window function partition by review_id, order.
-- by creation time. After the whole process, check whether each review is unique.
-- The review is unique after the whole process with 98411 row and 98411 distinct review_id.

WITH not_null_review_score AS(
  SELECT 
    *
  FROM
  order_reviews
  WHERE review_score IS NOT NULL
),
integer_review_score AS(
  SELECT 
    * 
  FROM 
  not_null_review_score 
  WHERE 
  TRY_CAST(review_score AS INT) IS NOT NULL),
not_null_review_id AS(
  SELECT * 
  FROM
  integer_review_score
  WHERE review_id IS NOT NULL
),
ranked AS(
  SELECT 
  not_null_review_id.*,
  ROW_NUMBER() OVER (PARTITION BY review_id ORDER BY review_creation_date DESC) AS rn
FROM not_null_review_id )
SELECT COUNT(*),COUNT(DISTINCT review_id) FROM ranked WHERE rn = 1


In [0]:
--Convert review_score to int, load all columns except rn to silver layer
CREATE OR REPLACE TABLE silver_olist.order_reviews AS
WITH not_null_review_score AS(
  SELECT 
    *
  FROM
  order_reviews
  WHERE review_score IS NOT NULL
),
integer_review_score AS(
  SELECT 
    * 
  FROM 
  not_null_review_score 
  WHERE 
  TRY_CAST(review_score AS INT) IS NOT NULL),
not_null_review_id AS(
  SELECT * 
  FROM
  integer_review_score
  WHERE review_id IS NOT NULL
),
ranked AS(
  SELECT 
  not_null_review_id.*,
  ROW_NUMBER() OVER (PARTITION BY review_id ORDER BY review_creation_date DESC) AS rn
FROM not_null_review_id 
)
SELECT 
    review_id,
    order_id,
    CAST(review_score AS INT) AS review_score,
    review_comment_title,
    review_comment_message,
    DATE(review_creation_date),
    CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_ts
FROM ranked
WHERE rn = 1;

#####5. Join products and product_category_name and load to silver layer

In [0]:
--View products table
select * from products limit 5

In [0]:
--Check if there is product with null value in product category name; and there are 610 rows. 
SELECT * FROM products WHERE product_category_name IS NULL

In [0]:
--View product_category_name table
SELECT * FROM product_category_name LIMIT 5

In [0]:
--Left join products on product_category_name to get product_category_name_english column, modify type of some columns in products table. Then, load to products table in silver layer
CREATE OR REPLACE TABLE silver_olist.products AS
  SELECT 
    p.product_id,
    COALESCE(p.product_category_name,'Not specified') AS product_category_name,
    COALESCE(INITCAP(REPLACE(TRIM(c.product_category_name_english),'_',' ')),'Not specified') AS product_category_name_english,
    CAST(p.product_name_lenght AS INT) AS product_name_lenght,
    CAST(p.product_description_lenght AS INT) AS product_description_lenght,
    CAST(p.product_photos_qty AS INT) AS product_photos_qty,
    CAST(p.product_weight_g AS INT) AS product_weight_g,
    CAST(p.product_length_cm AS INT) AS product_length_cm,
    CAST(p.product_height_cm AS INT) AS product_height_cm,
    CAST(p.product_width_cm AS INT) AS product_width_cm
  FROM products p
  LEFT JOIN product_category_name c
  ON p.product_category_name = c.product_category_name

#####6. Table sellers

In [0]:
--View sellers table
SELECT * FROM sellers limit 5

In [0]:
--Check if each seller id is unique
SELECT 
  COUNT(DISTINCT seller_id),
  COUNT(*),
  COUNT(DISTINCT seller_id)=COUNT(*) AS is_unique
FROM sellers
--The result shows that seller_id is unique for each row

In [0]:
-- Add sellers table to silver layer containing seller_id and seller_zip_code_prefix
CREATE OR REPLACE TABLE silver_olist.sellers AS 
SELECT 
  seller_id,
  seller_zip_code_prefix as zip_code_prefix,
  INITCAP(seller_city) as city,
  seller_state as state
FROM sellers

##### 7. Table customers 

In [0]:
--View table customer
SELECT * FROM customers limit 5

In [0]:
--Check if each customer_id is unique
SELECT 
  COUNT(DISTINCT customer_id),
  COUNT(*),
  COUNT(DISTINCT customer_id)=COUNT(*) AS is_unique
FROM customers
--The result shows that each customer_id is unique

In [0]:
--Modify city column, load all colunns of customers to silver layer
CREATE OR REPLACE TABLE silver_olist.customers AS
SELECT 
  customer_id,
  customer_unique_id,
  customer_zip_code_prefix as zip_code_prefix,
  INITCAP(customer_city) as city,
  customer_state as state

FROM customers